In [4]:
# Cell 1: Configuración e imports
import requests
import pandas as pd
import unicodedata
import time
from datetime import datetime, timezone

# --- CONFIGURACIÓN: ajustar antes de ejecutar ---
MOODLE_URL = "https://escueladesig.com.ar/clases"   # sin "/server.php"
TOKEN = "3490ed0c7cf26225130ab4f4027ad9a0"
INPUT_CSV = r"C:\Reporte\energia_alumnos.csv"     # ruta/archivo de entrada (CSV con ; y encabezado)
OUTPUT_CSV = "output.csv"   # archivo de salida
MOODLE_FORMAT = "json"
# --------------------------------------------------

# Helper: convertir epoch a string legible (zona local)
def epoch_to_str(ts):
    try:
        return datetime.fromtimestamp(int(ts), tz=timezone.utc).astimezone().strftime("%Y-%m-%d %H:%M:%S")
    except:
        return None

# Función base para llamar al WebService REST de Moodle
def call_moodle(wsfunction, params=None, method="post", timeout=30):
    """
    Llama a la API REST de Moodle.
    - wsfunction: nombre de la función (ej. 'core_user_get_users_by_field')
    - params: dict de parámetros adicionales (sin wstoken/wsfunction)
    - method: "get" o "post"
    Devuelve el JSON parseado o lanza excepción.
    """
    if params is None:
        params = {}
    url = f"{MOODLE_URL}/webservice/rest/server.php"
    base = {
        "wstoken": TOKEN,
        "wsfunction": wsfunction,
        "moodlewsrestformat": MOODLE_FORMAT
    }
    payload = {**base, **params}
    try:
        if method.lower() == "get":
            r = requests.get(url, params=payload, timeout=timeout)
        else:
            r = requests.post(url, data=payload, timeout=timeout)
        r.raise_for_status()
        return r.json()
    except Exception as e:
        # Re-raise with more context
        raise RuntimeError(f"Error calling Moodle ({wsfunction}): {e}")

In [5]:
# Cell 2: Funciones auxiliares para buscar usuarios, cursos, completitud y secciones

def clean_text(s):
    if not isinstance(s, str): return ""
    s = s.strip()
    s = unicodedata.normalize("NFKD", s)
    return "".join(c for c in s if not unicodedata.combining(c))

def find_users_by_username(username):
    try:
        res = call_moodle("core_user_get_users_by_field", {"field": "username", "values[0]": str(username)})
        if isinstance(res, list): return res
    except: return []

def find_users_by_idnumber(idnumber):
    try:
        res = call_moodle("core_user_get_users_by_field", {"field": "idnumber", "values[0]": str(idnumber)})
        if isinstance(res, list): return res
    except: return []

def find_users_by_name(firstname, lastname):
    q = f"{firstname} {lastname}".strip()
    if not q: return []
    try:
        params = {"criteria[0][key]": "query", "criteria[0][value]": q}
        res = call_moodle("core_user_get_users", params)
        if isinstance(res, dict) and res.get("users"): return res.get("users")
        if isinstance(res, list): return res
    except: return []

def get_user_courses(userid):
    try:
        res = call_moodle("core_enrol_get_users_courses", {"userid": userid})
        if isinstance(res, list): return res
    except: return []

def get_course_sections(courseid):
    try:
        res = call_moodle("core_course_get_contents", {"courseid": courseid})
        sections = {}
        if isinstance(res, list):
            for sec in res:
                sec_name = sec.get("name") or f"Sección {sec.get('section', 'N/A')}"
                for mod in sec.get("modules", []):
                    mod_id = mod.get("id")
                    if mod_id: sections[mod_id] = sec_name
        return sections
    except: return {}

def get_course_lastaccess(courseid, userid):
    try:
        res = call_moodle("core_user_get_course_user_profiles", {"courseid": courseid, "userid": userid})
        if isinstance(res, list) and len(res) > 0:
            item = res[0]
            for k in ("lastaccess", "lastlogin", "timeaccess"):
                if k in item and item.get(k): return epoch_to_str(item.get(k))
    except: pass
    return None

def get_course_completion_summary(courseid, userid):
    completed_count = 0
    total_activities = 0
    last_name = "Sin actividad"
    last_ts = 0
    sections_map = get_course_sections(courseid)

    try:
        res_act = call_moodle("core_completion_get_activities_completion_status", {"courseid": courseid, "userid": userid})
        activities = res_act.get("activities") or res_act.get("statuses") or []
        total_activities = len(activities)
        for a in activities:
            state = a.get("completionstate", None) or a.get("state", None)
            try:
                if state is not None and int(state) in (1, 2): completed_count += 1
            except: pass

            ts_candidates = [int(a.get(k, 0)) for k in ("timemodified", "timecompleted", "completiontime") if a.get(k)]
            ts = max(ts_candidates) if ts_candidates else 0
            if ts and ts > last_ts:
                last_ts = ts
                mod_name = a.get("name") or "Actividad"
                sec_name = sections_map.get(a.get("cmid"), "Sin sección")
                last_name = f"{sec_name} > {mod_name}"
    except: pass

    pct_num = int(round((completed_count / total_activities) * 100)) if total_activities > 0 else 0
    pct_str = f"{pct_num}%"
    last_time_str = epoch_to_str(last_ts) if last_ts > 0 else "Aún no comenzó"

    return (pct_str, last_name, last_time_str, completed_count, total_activities)

In [6]:
# Cell 3: Procesamiento principal

df_in = pd.read_csv(INPUT_CSV, sep=";", header=0, dtype=str, encoding="latin1").fillna("")
df_in.columns = [clean_text(c) for c in df_in.columns]

col_nombre, col_apellido, col_dni = "nombre", "apellido", "dni"
out_rows = []
total_rows = len(df_in)

print(f"🚀 Iniciando reporte para {total_rows} filas...")

for idx, row in df_in.iterrows():
    val_nombre, val_apellido = (row.get(col_nombre) or "").strip(), (row.get(col_apellido) or "").strip()
    val_dni = str(row.get(col_dni) or "").strip()

    print(f"\n[{idx+1}/{total_rows}] Analizando: {val_nombre} {val_apellido}")

    found_users = []
    if val_dni:
        found_users = find_users_by_username(val_dni) or find_users_by_idnumber(val_dni)
    if not found_users:
        found_users = find_users_by_name(val_nombre, val_apellido)

    if not found_users:
        out_rows.append({"DNI_CSV": val_dni, "Nombre_CSV": f"{val_nombre} {val_apellido}", "Status": "No encontrado"})
        continue

    for u in found_users:
        userid, fullname = u.get("id"), f"{u.get('firstname','')} {u.get('lastname','')}".strip()
        courses = get_user_courses(userid)
        
        if not courses:
            out_rows.append({"DNI_CSV": val_dni, "Nombre_Moodle": fullname, "Status": "Sin cursos"})
            continue

        for c in courses:
            cid, cname = c.get("id"), c.get("fullname")
            last_access = get_course_lastaccess(cid, userid)
            pct, last_mod, last_date, comp_count, total_act = get_course_completion_summary(cid, userid)

            # Lógica de fecha solicitada
            # Si Moodle no devuelve nada, ponemos "Aún no comenzó"
            fecha_final = last_date if last_date and last_date != "Sin registro" else (last_access if last_access else "Aún no comenzó")
            
            # Si no hay fecha de progreso pero hay acceso al curso, aclaramos que entró pero no avanzó
            if last_date == "Aún no comenzó" and last_access:
                last_mod = "Entró al curso pero no completó actividades"

            out_rows.append({
                "DNI_CSV": val_dni,
                "Nombre_Moodle": fullname,
                "Curso": cname,
                "Ultimo_Acceso_Curso": last_access if last_access else "Aún no comenzó",
                "Porcentaje_Progreso": pct,
                "Ultimo_Modulo_Visto": last_mod,
                "Fecha_Ultimo_Modulo": fecha_final,
                "Status": "OK"
            })
            print(f"     - {cname[:30]} | Progreso: {pct} | Fecha: {fecha_final}")

df_out = pd.DataFrame(out_rows)
df_out.to_csv(OUTPUT_CSV, index=False, encoding="utf-8-sig")
print(f"\n✅ Reporte generado: {OUTPUT_CSV}")
display(df_out.head(5))

🚀 Iniciando reporte para 22 filas...

[1/22] Analizando: Silvia Alaniz
     - Simbología y Mapas Temáticos 1 | Progreso: 84% | Fecha: 2026-05-07 22:16:56
     - Bases de Datos Espaciales | Progreso: 89% | Fecha: 2026-05-07 19:32:53
     - Herramientas para la Publicaci | Progreso: 86% | Fecha: 2026-05-07 19:30:59
     - Imágenes Satelitales | Progreso: 86% | Fecha: 2026-05-07 19:32:36
     - Introducción a Bases de Datos | Progreso: 85% | Fecha: 2026-05-07 20:25:42
     - QGIS Nivel II | Progreso: 91% | Fecha: 2026-05-07 22:03:46
     - QGIS Nivel I | Progreso: 88% | Fecha: 2026-05-07 21:26:13

[2/22] Analizando: Michelle Alves Ribeiro
     - Simbología y Mapas Temáticos 1 | Progreso: 0% | Fecha: Aún no comenzó
     - Bases de Datos Espaciales | Progreso: 0% | Fecha: Aún no comenzó
     - Herramientas para la Publicaci | Progreso: 0% | Fecha: Aún no comenzó
     - Imágenes Satelitales | Progreso: 0% | Fecha: Aún no comenzó
     - Introducción a Bases de Datos | Progreso: 0% | Fecha: Aú

,DNI_CSV,Nombre_Moodle,Curso,Ultimo_Acceso_Curso,Porcentaje_Progreso,Ultimo_Modulo_Visto,Fecha_Ultimo_Modulo,Status,Nombre_CSV
0,31138583,Silvia Alaniz,Simbología y Mapas Temáticos 1,Aún no comenzó,84%,¡Encuesta Final! > Actividad,2026-05-07 22:16:56,OK,NaN
1,31138583,Silvia Alaniz,Bases de Datos Espaciales,Aún no comenzó,89%,¡Encuesta Final! > Actividad,2026-05-07 19:32:53,OK,NaN
2,31138583,Silvia Alaniz,Herramientas para la Publicación de Mapas Web,Aún no comenzó,86%,¡Encuesta Final! > Actividad,2026-05-07 19:30:59,OK,NaN
3,31138583,Silvia Alaniz,Imágenes Satelitales,Aún no comenzó,86%,¡Encuesta Final! > Actividad,2026-05-07 19:32:36,OK,NaN
4,31138583,Silvia Alaniz,Introducción a Bases de Datos,Aún no comenzó,85%,¡Encuesta Final! > Actividad,2026-05-07 20:25:42,OK,NaN
